# WAF Cost Optimization
- Assessment Audit

This notebook assesses the Cost Optimization pillar using the **4 Key Topics** structure.

## Scoring Model (1-5 Scale)
| Score | Rating | Description |
|-------|--------|-------------|
| 5 | Excellent | Best practices fully implemented |
| 4 | Good | Most best practices, minor gaps |
| 3 | Moderate | Basic implementation |
| 2 | Limited | Minimal implementation |
| 1 | Poor | Not implemented |

## 4 Key Topics

1. **Business Value**
- ROI Analysis, Cost-Benefit Trade-offs, Value Attribution, Stakeholder Alignment

2. **Visibility**
- Cost Attribution, Usage Dashboards, Chargeback/Showback, Anomaly Detection

3. **Control**
- Resource Monitors, Warehouse Timeouts, Statement Timeouts, Governance Policies

4. **Optimization**
- Right-Sizing, Auto-Suspend Tuning, Storage Optimization, Query Efficiency, Serverless Features

In [7]:
SCHEMA = "IE"
ACCOUNT_ID = 100058
DATABASE = "SNOWHOUSE_IMPORT"
print(f"Target: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}")

Target: SNOWHOUSE_IMPORT.IE | Account ID: 100058


In [8]:
from snowflake.snowpark import Session
import pandas as pd
import os

connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse"
if 'session' not in globals():
    session = Session.builder.config("connection_name", connection_name).create()
print(f"Connected via: {connection_name}")

Connected via: snowhouse


---
# Key Topic 1: BUSINESS VALUE

**Connect costs to outcomes.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| ROI Analysis | No | Yes |
| Cost-Benefit Trade-offs | No | Yes |
| Value Attribution | No | Yes |
| Stakeholder Alignment | No | Yes |

## Manual Assessment: ROI Analysis

**Questions:**
1. Is ROI tracked for key workloads?
2. Is value measured against cost?
3. Are ROI metrics reported to leadership?

**Scoring:** 5=ROI tracked+reported | 3=Some tracking | 1=No tracking

**Manual Score:** _____

## Manual Assessment: Cost-Benefit Trade-offs

**Questions:**
1. Are speed vs cost decisions documented?
2. Are trade-offs reviewed?
3. Are decisions based on data?

**Scoring:** 5=Documented+data-driven | 3=Informal decisions | 1=No process

**Manual Score:** _____

## Manual Assessment: Value Attribution

**Questions:**
1. Is Snowflake spend tied to business outcomes?
2. Are business KPIs linked to data workloads?
3. Is value quantified?

**Scoring:** 5=Quantified value | 3=Some attribution | 1=No attribution

**Manual Score:** _____

## Manual Assessment: Stakeholder Alignment

**Questions:**
1. Do stakeholders understand their data costs?
2. Are SLA agreements in place?
3. Is there shared accountability?

**Scoring:** 5=Full alignment+accountability | 3=Partial awareness | 1=No alignment

**Manual Score:** _____

In [9]:
roi_score = 3; tradeoff_score = 3; value_score = 3; stakeholder_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 1: BUSINESS VALUE - Summary\n" + "="*60)
print(f"  ROI Analysis (Manual): {roi_score}/5\n  Cost-Benefit Trade-offs (Manual): {tradeoff_score}/5")
print(f"  Value Attribution (Manual): {value_score}/5\n  Stakeholder Alignment (Manual): {stakeholder_score}/5")
business_value_score = round((roi_score + tradeoff_score + value_score + stakeholder_score) / 4, 1)
print(f"\n📊 BUSINESS VALUE TOPIC SCORE: {business_value_score}/5")

KEY TOPIC 1: BUSINESS VALUE - Summary
  ROI Analysis (Manual): 3/5
  Cost-Benefit Trade-offs (Manual): 3/5
  Value Attribution (Manual): 3/5
  Stakeholder Alignment (Manual): 3/5

📊 BUSINESS VALUE TOPIC SCORE: 3.0/5


---
# Key Topic 2: VISIBILITY

**Know where your spend is going.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Cost Attribution | Yes |
- |
| Usage Dashboards | No | Yes |
| Chargeback/Showback | No | Yes |
| Anomaly Detection | Partial | Yes |

In [10]:
# Cost Attribution via Tags
tags_df = session.sql(f"""
SELECT COUNT(CASE WHEN UPPER(NAME) LIKE '%COST%CENTER%' OR UPPER(NAME) LIKE '%BUDGET%' OR UPPER(NAME) LIKE '%DEPARTMENT%' THEN ID END) as cost_tags
FROM {DATABASE}.{SCHEMA}.TAG_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
cost_tags = tags_df['COST_TAGS'].iloc[0]
print(f"=== Cost Attribution ===\nCost-related Tags: {cost_tags}")
cost_attr_score = 5 if cost_tags >= 3 else 4 if cost_tags >= 2 else 3 if cost_tags >= 1 else 1
print(f"Score: {cost_attr_score}/5")

=== Cost Attribution ===
Cost-related Tags: 0
Score: 1/5


## Manual Assessment: Usage Dashboards

**Questions:**
1. Are cost dashboards in place?
2. Are they reviewed regularly?
3. Are trends tracked?

**Scoring:** 5=Comprehensive dashboards | 3=Basic dashboards | 1=No dashboards

**Manual Score:** _____

## Manual Assessment: Chargeback/Showback

**Questions:**
1. Is there a chargeback process?
2. Are teams accountable for consumption?
3. Is there budget ownership?

**Scoring:** 5=Full chargeback | 3=Showback only | 1=No process

**Manual Score:** _____

## Manual Assessment: Anomaly Detection

**Questions:**
1. Are cost spikes investigated?
2. Are alerts configured?
3. Is there a response process?

**Scoring:** 5=Proactive detection | 3=Reactive investigation | 1=No detection

**Manual Score:** _____

In [11]:
dashboard_score = 3; chargeback_score = 3; anomaly_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 2: VISIBILITY - Summary\n" + "="*60)
print(f"  Cost Attribution (DDM): {cost_attr_score}/5\n  Usage Dashboards (Manual): {dashboard_score}/5")
print(f"  Chargeback/Showback (Manual): {chargeback_score}/5\n  Anomaly Detection (Manual): {anomaly_score}/5")
visibility_score = round((cost_attr_score + dashboard_score + chargeback_score + anomaly_score) / 4, 1)
print(f"\n📊 VISIBILITY TOPIC SCORE: {visibility_score}/5")

KEY TOPIC 2: VISIBILITY - Summary
  Cost Attribution (DDM): 1/5
  Usage Dashboards (Manual): 3/5
  Chargeback/Showback (Manual): 3/5
  Anomaly Detection (Manual): 3/5

📊 VISIBILITY TOPIC SCORE: 2.5/5


---
# Key Topic 3: CONTROL

**Prevent runaway costs.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Resource Monitors | Yes | RESOURCE_MONITOR_ETL_V |
| Warehouse Timeouts | Yes | WAREHOUSE_ETL_V |
| Statement Timeouts | Partial | WAREHOUSE_ETL_V |
| Governance Policies | No | Manual |

In [12]:
# Resource Monitors
rm_df = session.sql(f"""
SELECT COUNT(*) as rm_count
FROM {DATABASE}.{SCHEMA}.RESOURCE_MONITOR_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
rm_count = rm_df['RM_COUNT'].iloc[0]
print(f"=== Resource Monitors ===\nResource Monitors: {rm_count}")
rm_score = 5 if rm_count >= 3 else 4 if rm_count >= 2 else 3 if rm_count >= 1 else 1
print(f"Score: {rm_score}/5")

=== Resource Monitors ===
Resource Monitors: 0
Score: 1/5


In [15]:
# Warehouse Auto-Suspend Analysis
wh_df = session.sql(f"""
SELECT COUNT(*) as total_wh
FROM {DATABASE}.{SCHEMA}.WAREHOUSE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
total_wh = wh_df['TOTAL_WH'].iloc[0]
print(f"=== Warehouse Configuration ===\nTotal Warehouses: {total_wh}")
print("Note: Auto-suspend settings not available in DDM - verify manually")
timeout_score = 3  # Requires manual verification
print(f"Score: {timeout_score}/5 (requires manual verification)")

=== Warehouse Configuration ===
Total Warehouses: 157
Note: Auto-suspend settings not available in DDM - verify manually
Score: 3/5 (requires manual verification)


## Manual Assessment: Governance Policies

**Questions:**
1. Is approval required for new warehouses?
2. Are size increases reviewed?
3. Is there a governance process?

**Scoring:** 5=Full governance | 3=Informal process | 1=No governance

**Manual Score:** _____

In [16]:
governance_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 3: CONTROL - Summary\n" + "="*60)
print(f"  Resource Monitors (DDM): {rm_score}/5\n  Warehouse Timeouts (DDM): {timeout_score}/5")
print(f"  Governance Policies (Manual): {governance_score}/5")
control_score = round((rm_score + timeout_score + governance_score) / 3, 1)
print(f"\n📊 CONTROL TOPIC SCORE: {control_score}/5")

KEY TOPIC 3: CONTROL - Summary
  Resource Monitors (DDM): 1/5
  Warehouse Timeouts (DDM): 3/5
  Governance Policies (Manual): 3/5

📊 CONTROL TOPIC SCORE: 2.3/5


---
# Key Topic 4: OPTIMIZATION

**Reduce costs without sacrificing value.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Right-Sizing | Partial | WAREHOUSE_ETL_V |
| Auto-Suspend Tuning | Yes | WAREHOUSE_ETL_V |
| Storage Optimization | Partial | TABLE_ETL_V, DATABASE_ETL_V |
| Query Efficiency | No | Manual |
| Serverless Features | Yes | USER_TASK_ETL_V, PIPE_ETL_V |

In [17]:
# Storage Optimization - Transient tables
storage_df = session.sql(f"""
SELECT COUNT(*) as total_tables,
    COUNT(CASE WHEN DATA_TRANSIENT = TRUE THEN 1 END) as transient_tables
FROM {DATABASE}.{SCHEMA}.TABLE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND KIND_ID = 1
""").to_pandas()
transient = storage_df['TRANSIENT_TABLES'].iloc[0]
print(f"=== Storage Optimization ===\nTransient Tables (staging): {transient}")
storage_score = 5 if transient >= 50 else 4 if transient >= 20 else 3 if transient >= 5 else 2 if transient >= 1 else 1
print(f"Score: {storage_score}/5")

=== Storage Optimization ===
Transient Tables (staging): 22543
Score: 5/5


In [18]:
# Serverless Features
serverless_df = session.sql(f"""
SELECT
    (SELECT COUNT(*) FROM {DATABASE}.{SCHEMA}.USER_TASK_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND WAREHOUSE_NAME IS NULL) as serverless_tasks,
    (SELECT COUNT(*) FROM {DATABASE}.{SCHEMA}.PIPE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL) as pipes
""").to_pandas()
serverless_tasks = serverless_df['SERVERLESS_TASKS'].iloc[0]
pipes = serverless_df['PIPES'].iloc[0]
print(f"=== Serverless Features ===\nServerless Tasks: {serverless_tasks}\nPipes: {pipes}")
serverless_total = serverless_tasks + pipes
serverless_score = 5 if serverless_total >= 20 else 4 if serverless_total >= 10 else 3 if serverless_total >= 5 else 2 if serverless_total >= 1 else 1
print(f"Score: {serverless_score}/5")

=== Serverless Features ===
Serverless Tasks: 298
Pipes: 3680
Score: 5/5


## Manual Assessment: Right-Sizing

**Questions:**
1. Are warehouses regularly reviewed for right-sizing?
2. Is sizing based on workload analysis?
3. Are recommendations implemented?

**Scoring:** 5=Regular reviews+action | 3=Occasional reviews | 1=No reviews

**Manual Score:** _____

## Manual Assessment: Query Efficiency

**Questions:**
1. Are expensive queries optimized?
2. Is there a query review process?
3. Are inefficient patterns addressed?

**Scoring:** 5=Systematic optimization | 3=Ad-hoc optimization | 1=No optimization

**Manual Score:** _____

In [19]:
rightsizing_score = 3; query_eff_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 4: OPTIMIZATION - Summary\n" + "="*60)
print(f"  Right-Sizing (Manual): {rightsizing_score}/5\n  Auto-Suspend (DDM): {timeout_score}/5")
print(f"  Storage Optimization (DDM): {storage_score}/5\n  Query Efficiency (Manual): {query_eff_score}/5\n  Serverless Features (DDM): {serverless_score}/5")
optimization_score = round((rightsizing_score + timeout_score + storage_score + query_eff_score + serverless_score) / 5, 1)
print(f"\n📊 OPTIMIZATION TOPIC SCORE: {optimization_score}/5")

KEY TOPIC 4: OPTIMIZATION - Summary
  Right-Sizing (Manual): 3/5
  Auto-Suspend (DDM): 3/5
  Storage Optimization (DDM): 5/5
  Query Efficiency (Manual): 3/5
  Serverless Features (DDM): 5/5

📊 OPTIMIZATION TOPIC SCORE: 3.8/5


---
# Overall Assessment Summary

In [20]:
print("="*70 + "\nCOST OPTIMIZATION - OVERALL ASSESSMENT\n" + "="*70)
topics = {"1. Business Value": business_value_score, "2. Visibility": visibility_score, "3. Control": control_score, "4. Optimization": optimization_score}
for topic, score in topics.items():
    bar = "█" * int(score) + "░" * (5 - int(score))
    status = "✅" if score >= 4 else "⚠️" if score >= 3 else "❌"
    print(f"{status} {topic}: {bar} {score}/5")
overall_score = round(sum(topics.values()) / len(topics), 1)
print(f"\n🏆 OVERALL COST OPTIMIZATION SCORE: {overall_score}/5")

COST OPTIMIZATION - OVERALL ASSESSMENT
⚠️ 1. Business Value: ███░░ 3.0/5
❌ 2. Visibility: ██░░░ 2.5/5
❌ 3. Control: ██░░░ 2.3/5
⚠️ 4. Optimization: ███░░ 3.8/5

🏆 OVERALL COST OPTIMIZATION SCORE: 2.9/5


In [21]:
prompt = f"""WAF Cost Optimization advisor. Provide:
1. EXECUTIVE SUMMARY (2-3 sentences)
2. CRITICAL FINDINGS (3-5 issues with priority)
3. QUICK WINS (3-5 actions for 1 week)

Cost Optimization: {overall_score}/5 | Topics: {topics}"""
result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', '{prompt.replace(chr(39), chr(39)+chr(39))}') as response").collect()
print("🤖 AI INSIGHTS\n" + result[0]['RESPONSE'])

🤖 AI INSIGHTS
WAF COST OPTIMIZATION ASSESSMENT

1. EXECUTIVE SUMMARY
The Web Application Firewall (WAF) implementation shows moderate cost efficiency with a score of 2.9/5, indicating significant room for improvement. While optimization practices are strong (3.8), the environment lacks adequate cost controls (2.3) and visibility (2.5), potentially leading to unnecessary expenses and inefficient resource utilization.

2. CRITICAL FINDINGS
🔴 HIGH: Insufficient cost monitoring and alerting mechanisms for WAF usage spikes
🔴 HIGH: Lack of automated response to abnormal traffic patterns causing excessive WAF processing
🟡 MEDIUM: Poor visibility into rule-based cost attribution and impact
🟡 MEDIUM: Missing business-aligned cost allocation strategy

3. QUICK WINS
✅ Implement WAF cost monitoring dashboard with daily usage metrics and alerts
✅ Enable rate limiting for top consuming endpoints to prevent cost spikes
✅ Review and disable unused WAF rules to reduce processing overhead
✅ Tag WAF reso

---
## Coverage Matrix

| Topic | Sub-Topic | DDM | Manual | Source |
|-------|-----------|-----|--------|--------|
| Business Value | ROI Analysis |
- | ✅ | N/A |
| Business Value | Cost-Benefit Trade-offs |
- | ✅ | N/A |
| Business Value | Value Attribution |
- | ✅ | N/A |
| Business Value | Stakeholder Alignment |
- | ✅ | N/A |
| Visibility | Cost Attribution | ✅ |
- | TAG_ETL_V |
| Visibility | Usage Dashboards |
- | ✅ | N/A |
| Visibility | Chargeback/Showback |
- | ✅ | N/A |
| Visibility | Anomaly Detection |
- | ✅ | N/A |
| Control | Resource Monitors | ✅ |
- | RESOURCE_MONITOR_ETL_V |
| Control | Warehouse Timeouts | ✅ |
- | WAREHOUSE_ETL_V |
| Control | Governance Policies |
- | ✅ | N/A |
| Optimization | Right-Sizing |
- | ✅ | N/A |
| Optimization | Auto-Suspend | ✅ |
- | WAREHOUSE_ETL_V |
| Optimization | Storage Optimization | ✅ |
- | TABLE_ETL_V |
| Optimization | Query Efficiency |
- | ✅ | N/A |
| Optimization | Serverless Features | ✅ |
- | TASK/PIPE_ETL_V |

**Summary:** 6 DDM | 10 Manual Only